# Retriever backbone comparison

Qualitative side-by-side of DINOv3 / ResNet-50 / SigLIP-2 / ViT as per-frame retrievers.

**Prereqs (run once):**
1. `python scripts/data_collection/build_all_backbones.py` — builds FAISS caches.
2. `python scripts/data_collection/extract_fixed_frames.py` — extracts fixed query frames.

See `docs/superpowers/specs/2026-05-25-retriever-backbone-comparison-design.md` for design context.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # repo root

from PIL import Image
import matplotlib.pyplot as plt

from hanoi_caption.retrieval.backbones import (
    Dinov3Extractor, Resnet50Extractor, Siglip2Extractor, VitExtractor,
)
from hanoi_caption.retrieval.index import build_or_load_index
from hanoi_caption.retrieval.retrieve import make_retrieve_fn, make_topk_fn
from hanoi_caption.video_pipeline import sample_frames

KB_DIR         = Path('../data/kb_images')
CACHE_DIR      = Path('../data/cache')
FIXED_FRAMES   = Path('../tests/fixtures/retriever_frames')
TIMELINE_VIDEO = Path('../tests/videos/NhaThoLon_S_T03.MOV')
TOPK = 5
SAMPLE_FPS = 1.0


In [ ]:
EXTRACTORS = {
    'dinov3':   Dinov3Extractor(),
    'resnet50': Resnet50Extractor(),
    'siglip2':  Siglip2Extractor(),
    'vit':      VitExtractor(),
}
INDEXES = {n: build_or_load_index(e, KB_DIR, CACHE_DIR) for n, e in EXTRACTORS.items()}
for n, (idx, _) in INDEXES.items():
    print(f'{n:10s} dim={EXTRACTORS[n].dim:4d}  vectors={idx.ntotal}')


In [ ]:
# Section A helper: top-K grid
def show_topk_grid(queries, topk_by_model, cell_size=(2.0, 2.0)):
    model_names = list(topk_by_model.keys())
    k = len(next(iter(topk_by_model.values()))[0])
    n_rows = len(queries)
    n_cols = 1 + len(model_names) * k
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(cell_size[0] * n_cols, cell_size[1] * n_rows),
    )
    if n_rows == 1:
        axes = axes[None, :]
    for r, (label, qimg) in enumerate(queries):
        axes[r, 0].imshow(qimg)
        axes[r, 0].set_title(f'Q: {label}', fontsize=8)
        axes[r, 0].axis('off')
        c = 1
        for mname in model_names:
            for rank, res in enumerate(topk_by_model[mname][r]):
                ax = axes[r, c]
                ax.imshow(Image.open(res['path']))
                if rank == 0:
                    ax.set_title(f'{mname}\n{res["kb_id"]}\n{res["score"]:.2f}', fontsize=7)
                else:
                    ax.set_title(f'{res["kb_id"]}\n{res["score"]:.2f}', fontsize=7)
                ax.axis('off')
                c += 1
    plt.tight_layout()
    return fig

# Section B helper: timeline
def show_timeline(timeline_by_model, sample_fps=1.0):
    model_names = list(timeline_by_model.keys())
    all_kbs = sorted({kb for tl in timeline_by_model.values() for _, kb, _ in tl if kb})
    cmap = plt.cm.tab20
    palette = {kb: list(cmap(i % 20)) for i, kb in enumerate(all_kbs)}
    UNKNOWN = (0.85, 0.85, 0.85, 1.0)
    stride = 1.0 / sample_fps
    fig, ax = plt.subplots(figsize=(14, 1.0 * len(model_names) + 1))
    for row, mname in enumerate(model_names):
        for t, kb, score in timeline_by_model[mname]:
            color = list(palette.get(kb, UNKNOWN))
            color[3] = min(1.0, max(0.3, float(score)))
            ax.barh(row, stride, left=t, height=0.8, color=color, edgecolor='none')
    ax.set_yticks(range(len(model_names)))
    ax.set_yticklabels(model_names)
    ax.set_xlabel('time (s)')
    handles = [plt.Line2D([0], [0], color=palette[k], lw=6, label=k) for k in all_kbs]
    ax.legend(handles=handles, bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
    plt.tight_layout()
    return fig


## Section A — Top-K view
For each fixed query frame, show the top-5 nearest KB images per backbone.

In [ ]:
fixed_paths = sorted(FIXED_FRAMES.glob('*.jpg'))
assert fixed_paths, f'No fixtures at {FIXED_FRAMES} - run extract_fixed_frames.py first'
queries = [(p.stem, Image.open(p)) for p in fixed_paths]

topk_results = {}
for name, ext in EXTRACTORS.items():
    fn = make_topk_fn(ext, *INDEXES[name], k=TOPK)
    topk_results[name] = [fn(img) for _, img in queries]

_ = show_topk_grid(queries, topk_results, cell_size=(3.5, 3.5))


## Section B - Timeline view
Sample `TIMELINE_VIDEO` at 1 fps and plot per-frame predicted `kb_id` (colored) with score as opacity, one band per backbone.

In [ ]:
assert TIMELINE_VIDEO.exists(), f'Missing {TIMELINE_VIDEO}'
sampled = sample_frames(TIMELINE_VIDEO, sample_fps=SAMPLE_FPS)
frames = [img for _, _, img in sampled]
times  = [t   for _, t, _   in sampled]
print(f'sampled {len(frames)} frames from {TIMELINE_VIDEO.name}')

timeline_by_model = {}
for name, ext in EXTRACTORS.items():
    fn = make_retrieve_fn(ext, *INDEXES[name])
    timeline_by_model[name] = [(t, *fn(img)) for img, t in zip(frames, times)]

_ = show_timeline(timeline_by_model, sample_fps=SAMPLE_FPS)


## Section C - Quick stats
Coarse signal until ground-truth segmentation lands. Latency is per-frame on CUDA, averaged over the timeline frames; `frac score > 0.5` is per-backbone (cosine scale is *not* strictly cross-backbone comparable).

In [ ]:
import time

rows = []
for name, ext in EXTRACTORS.items():
    fn = make_retrieve_fn(ext, *INDEXES[name])
    # warm-up
    fn(frames[0])
    t0 = time.perf_counter()
    preds = [fn(img) for img in frames]
    dt = (time.perf_counter() - t0) / len(frames) * 1000
    high = sum(1 for kb, sc in preds if sc > 0.5)
    rows.append((name, ext.dim, f'{dt:6.1f} ms', f'{high/len(preds):.0%}'))

print(f'{"model":10s} {"dim":>5s} {"latency/frame":>14s} {"frac score>0.5":>16s}')
print('-' * 50)
for r in rows:
    print(f'{r[0]:10s} {r[1]:5d} {r[2]:>14s} {r[3]:>16s}')


## Section D - Eval metrics comparison
Loads `../data/eval/seg_metrics_<backbone>.json` for every backbone that has been run and renders one sortable table + a grouped F1 bar chart. Rows sorted by F1@0.3 descending; missing files are silently skipped.

In [ ]:
import json
import pandas as pd
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # repo root

EVAL_DIR = Path('../data/eval')
THRESHOLDS = ['0.3', '0.5', '0.7']
SHORT = {'precision': 'P', 'recall': 'R', 'f1': 'F1', 'lid_acc': 'LID'}

# (backbone_name, params_label) — order is display-only; table is re-sorted by F1@0.3
BACKBONES = [
    ('siglip2_large',  '~400M'),
    ('dinov3_vitl16',  '300M'),
    ('aimv2_large',    '~300M'),
    ('dinov3_vits16',  '22M'),
    ('siglip2_base',   '~200M'),
    ('vit_base',       '86M'),
    ('resnet50',       '25M'),
]

rows = []
for name, params in BACKBONES:
    p = EVAL_DIR / f'seg_metrics_{name}.json'
    if not p.exists():
        continue
    thr = json.loads(p.read_text(encoding='utf-8'))['thresholds']
    row = {'params': params}
    for t in THRESHOLDS:
        for k, short in SHORT.items():
            row[f'{short}@{t}'] = thr[t][k]
    rows.append((name, row))

df = pd.DataFrame([r for _, r in rows], index=[n for n, _ in rows])
df.index.name = 'backbone'
df = df.sort_values('F1@0.3', ascending=False)

numeric_cols = [c for c in df.columns if c != 'params']
df.style.background_gradient(
    subset=numeric_cols, cmap='RdYlGn', vmin=0.5, vmax=1.0
).format('{:.3f}', subset=numeric_cols)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

f1_cols = [f'F1@{t}' for t in THRESHOLDS]
lid_cols = [f'LID@{t}' for t in THRESHOLDS]
backbones = df.index.tolist()
x = np.arange(len(backbones))
w = 0.25

fig, (ax_f1, ax_lid) = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for ax, cols, title in [(ax_f1, f1_cols, 'F1'), (ax_lid, lid_cols, 'LID-Acc')]:
    for i, c in enumerate(cols):
        ax.bar(x + (i - 1) * w, df[c].values, w, label=c)
    ax.set_xticks(x)
    ax.set_xticklabels(backbones, rotation=30, ha='right')
    ax.set_ylim(0.4, 1.02)
    ax.grid(axis='y', alpha=0.3)
    ax.legend(loc='lower left')
    ax.set_title(f'{title} per backbone across TIoU thresholds')
ax_f1.set_ylabel('score')
plt.tight_layout()
plt.show()


## Section E - Per-clip segment timeline

Side-by-side timeline view: ground-truth segments vs each backbone's predicted segments for a chosen video. Reads `../data/eval/pipeline_results_<backbone>.json` (produced by `run_module2.py`) and `../data/eval/test_set.json`. Two modes below; pick or run both.

- **Bar opacity** = confidence (GT is always solid 1.0).
- **Bar color** = `kb_id`, palette consistent within one figure.
- Backbones without a pipeline_results file are silently skipped per video.

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

EVAL_DIR_E = Path('../data/eval')

# Reuse BACKBONES from Section D if available; otherwise define a default list.
try:
    BACKBONES
except NameError:
    BACKBONES = [
        ('siglip2_large',  '~400M'), ('dinov3_vitl16', '300M'),
        ('aimv2_large',    '~300M'), ('dinov3_vits16', '22M'),
        ('siglip2_base',   '~200M'), ('vit_base',      '86M'),
        ('resnet50',       '25M'),
    ]

def _load_test_set():
    return json.loads((EVAL_DIR_E / 'test_set.json').read_text(encoding='utf-8'))

def _load_all_results():
    out = {}
    for name, _ in BACKBONES:
        p = EVAL_DIR_E / f'pipeline_results_{name}.json'
        if p.exists():
            out[name] = {r['video_id']: r for r in json.loads(p.read_text(encoding='utf-8'))}
    return out

def _palette(kbs):
    cmap = plt.cm.tab20
    return {k: list(cmap(i % 20)) for i, k in enumerate(sorted(kbs))}

def plot_video_timeline(filename, backbones_order, test_set, results_by_backbone, ax=None):
    """One figure: rows = GT + each backbone (that has data for this video), x = time."""
    video = next((v for v in test_set['in_kb'] if v['filename'] == filename), None)
    if video is None:
        print(f'video not found in test_set: {filename}')
        return
    duration = video['duration']
    gt_segs = video['gt_segments']

    avail = [b for b in backbones_order
             if b in results_by_backbone and video['video_id'] in results_by_backbone[b]]

    kbs = {s['kb_id'] for s in gt_segs}
    for b in avail:
        for s in results_by_backbone[b][video['video_id']].get('predicted_segments', []):
            if s.get('kb_id'):
                kbs.add(s['kb_id'])
    pal = _palette(kbs)
    rows = ['GT'] + avail

    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(14, 0.5 * len(rows) + 1.2))

    for i, row_name in enumerate(rows):
        if row_name == 'GT':
            segs = [(s['start_time'], s['end_time'], s['kb_id'], 1.0) for s in gt_segs]
        else:
            segs = [(s['start_s'], s['end_s'], s['kb_id'], s['confidence'])
                    for s in results_by_backbone[row_name][video['video_id']].get('predicted_segments', [])]
        for start, end, kb, conf in segs:
            base = list(pal.get(kb, [0.85, 0.85, 0.85, 1.0]))
            color = base[:3] + [min(1.0, max(0.3, float(conf)))]
            ax.barh(i, end - start, left=start, height=0.7,
                    color=color, edgecolor='black', linewidth=0.3)

    ax.set_yticks(range(len(rows)))
    ax.set_yticklabels(rows, fontsize=9)
    ax.invert_yaxis()
    ax.set_xlim(0, duration)
    ax.set_xlabel('time (s)')
    ax.set_title(filename, fontsize=10)

    handles = [mpatches.Patch(color=pal[k], label=k) for k in sorted(kbs)]
    ax.legend(handles=handles, bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=7)
    if own_fig:
        plt.tight_layout()
    return ax

ts = _load_test_set()
results = _load_all_results()
all_filenames = [v['filename'] for v in ts['in_kb']]
print(f'test_set: {len(ts["in_kb"])} videos    backbones with results: {sorted(results)}')
print('\nFirst 10 filenames (copy-paste into VIDEO_FILENAME below):')
for f in all_filenames[:10]:
    print(f'  {f}')


In [ ]:
# === Mode 1: single video ===
# Edit VIDEO_FILENAME to inspect a specific clip. The default uses the first one in test_set.

VIDEO_FILENAME = all_filenames[7]
# e.g. VIDEO_FILENAME = 'A1_018_DenNgocSonToanCanh_M_T02.mp4'

plot_video_timeline(VIDEO_FILENAME, [n for n, _ in BACKBONES], ts, results)
plt.show()


In [ ]:
# === Mode 2: all videos ===
# Renders one figure per video. With many clips this output gets long;
# cap with MAX_VIDEOS (None = no limit).
MAX_VIDEOS = 10

videos_to_show = ts['in_kb'][:MAX_VIDEOS] if MAX_VIDEOS else ts['in_kb']
for video in videos_to_show:
    plot_video_timeline(video['filename'], [n for n, _ in BACKBONES], ts, results)
    plt.show()
